## 

## Vendas consolidadas

In [0]:
%sql
-- Criar a Tabela Ouro de Vendas Consolidadas (Receita Bruta).
-- Regras aplicadas:
-- INNER JOIN entre Fato (Vendas) e Dimensões (Clientes e Produtos).
-- ==============================================================================

CREATE OR REPLACE TABLE lh_nautical.lh_nautical_db.gld_vendas_consolidadas AS
SELECT 
    v.id_sale,
    v.id_client,
    v.sale_date_clean AS sale_date,
    v.id_product,
    
    -- Trazendo dados do cliente
    c.client_name,
    c.state AS client_state,
    
    -- Trazendo dados do Produto
    p.original_name AS product_name,
    p.category_clean AS product_category,
    
    -- Métricas da Venda
    v.qtd AS quantity,
    v.total_num AS gross_revenue
    
FROM lh_nautical.lh_nautical_db.slv_vendas v
INNER JOIN lh_nautical.lh_nautical_db.slv_produtos p 
    ON v.id_product = p.id_product
INNER JOIN lh_nautical.lh_nautical_db.slv_clientes c 
    ON v.id_client = c.id_client;

-- Validação
SELECT * FROM lh_nautical.lh_nautical_db.gld_vendas_consolidadas LIMIT 10;

id_sale,id_client,sale_date,id_product,client_name,client_state,product_name,product_category,quantity,gross_revenue
11,39,2023-05-07,128,Renata Oliveira Siqueira,PE,Boia de Arqueamento Bruce Marlin Hydra,ANCORAGEM,5,23254.0
24,45,2023-05-09,128,Amanda Santos Figueiredo Rodrigues Siqueira,PA,Boia de Arqueamento Bruce Marlin Hydra,ANCORAGEM,12,55809.0
27,23,2023-04-15,97,Bruna Mendonça Rodrigues,SP,Motor de Popa Tohatsu Evo 168HP,PROPULSAO,4,541340.0
51,2,2024-08-28,52,Fernanda Azevedo Soares Nunes Vieira,PE,Motor Diesel Honda Zen Tidal Mako 26HP,PROPULSAO,4,54816.0
79,10,2024-12-31,12,Débora Paiva,PA,GPS AIS Drift Flux Vox,ELETRONICOS,13,35440.0
81,28,2024-04-11,82,Bianca Rodrigues,PR,Motor Elétrico Volvo Nautic Vox 118HP,PROPULSAO,10,490523.0
126,29,2024-07-14,80,Gabriela Moraes Macedo Nunes,CE,Motor de Popa Volvo Maré 69HP,PROPULSAO,5,613813.05
133,34,2024-11-13,122,Victor Vieira Nunes Cunha,CE,Corrente Danforth Zenith Oceanic Torque,ANCORAGEM,2,2512.75
141,29,2024-06-15,127,Gabriela Moraes Macedo Nunes,CE,Cabo de Nylon Delta Velocity Core Mako,ANCORAGEM,5,7359.65
157,48,2024-04-24,17,Letícia Torres Peixoto Oliveira,PE,Rádio AIS Oceanic Pulse,ELETRONICOS,1,13337.05


## Lucratividade

In [0]:
%sql
CREATE OR REPLACE TABLE lh_nautical.lh_nautical_db.gld_lucro_detalhado AS
WITH cte_vendas_com_historico AS (
    SELECT 
        v.id_sale,
        v.id_client,
        v.sale_date,
        v.client_name,
        v.client_state,
        v.product_category,
        v.product_name,
        v.quantity,
        v.gross_revenue,
        c.unit_cost_usd,
        
        ROW_NUMBER() OVER(
            PARTITION BY v.id_sale 
            ORDER BY c.start_date_clean DESC 
        ) as rank_custo
        
    FROM lh_nautical.lh_nautical_db.gld_vendas_consolidadas v
    LEFT JOIN lh_nautical.lh_nautical_db.slv_custos c
        ON v.id_product = c.product_id 
        AND c.start_date_clean <= v.sale_date
)
SELECT 
    id_sale,
    id_client,
    sale_date,
    client_name,
    client_state,
    product_category,
    product_name,
    quantity,
    gross_revenue,
    ROUND((unit_cost_usd * quantity), 2) AS total_cost,
    ROUND((gross_revenue - (unit_cost_usd * quantity)), 2) AS gross_profit
FROM cte_vendas_com_historico
WHERE rank_custo = 1;

SELECT * FROM lh_nautical.lh_nautical_db.gld_lucro_detalhado LIMIT 10

id_sale,id_client,sale_date,client_name,client_state,product_category,product_name,quantity,gross_revenue,total_cost,gross_profit
0,42,2023-09-10,Márcia Figueiredo,PA,ANCORAGEM,Cabo de Nylon Danforth Prime,11,3405.0,689.04,2715.96
1,3,2024-09-15,Daniel Farias Ribeiro Teixeira,RS,ANCORAGEM,Cabo de Nylon Bruce Flux Hydro,9,16873.9,3515.58,13358.32
2,25,2024-08-13,Femininos Antunes Lopes Ribeiro Amaral,ES,ANCORAGEM,Boia de Arqueamento Danforth Torque,7,9475.3,1728.23,7747.07
4,20,2023-02-03,Bruno Silva,PA,ELETRONICOS,Piloto Automático Furuno Torque Peak,5,55893.0,10543.7,45349.3
5,8,2024-02-12,Luiz Alves Pimentel,SE,PROPULSAO,Motor de Popa Honda Vector Kinetic 174HP,4,451403.9,89991.16,361412.74
6,36,2023-09-26,Francisca Ribeiro Pinheiro,PA,PROPULSAO,Motor Diesel Honda Zen Tidal Mako 26HP,3,39056.4,7904.07,31152.33
8,27,2024-02-28,Flávia Nunes Martins Ribeiro Coelho,BA,ELETRONICOS,Rádio Lowrance Nitro Thrust Barracuda,3,34560.05,7435.44,27124.61
9,37,2023-11-07,Jéssica Pimentel Borges Nunes Teixeira,MA,ELETRONICOS,Piloto Automático Furuno Core Boost Flux,7,114932.9,23890.16,91042.74
10,31,2024-08-25,Jéssica Figueiredo Leite Martins,MA,ANCORAGEM,Âncora Delta Hydra Kraken Velocity,3,12643.55,2690.49,9953.06
11,39,2023-05-07,Renata Oliveira Siqueira,PE,ANCORAGEM,Boia de Arqueamento Bruce Marlin Hydra,5,23254.0,4265.45,18988.55
